# CONCH + Camelyon16-BMAD (ECR4AD)
*Anomaly detection on Colab; requires a GPU runtime and a Hugging Face token with MahmoodLab/conch access.*

# GPU
*Verify accelerator availability.*

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-3c9e5741-7894-434d-d360-ffeb59421a26)


# Code
*Clone the repository.*

In [2]:
import os

REPO = '/content/wsi_anomaly_detection'
if not os.path.isdir(REPO):
    !git clone https://github.com/ARHPA/ECR4AD-BMAD-Reproduction.git {REPO}

os.chdir(REPO)
os.environ['PYTHONPATH'] = os.pathsep.join([REPO, os.path.join(REPO, 'networks')])

Cloning into '/content/wsi_anomaly_detection'...
remote: Enumerating objects: 372, done.
remote: Counting objects: 100% (372/372), done.
remote: Compressing objects: 100% (253/253), done.
remote: Total 372 (delta 110), reused 370 (delta 108), pack-reused 0 (from 0)
Receiving objects: 100% (372/372), 9.54 MiB | 16.15 MiB/s, done.
Resolving deltas: 100% (110/110), done.


# Dependencies
*Install dependencies.*

In [3]:
!pip -q install timm==0.9.16 transformers huggingface_hub wandb gdown pyyaml scikit-learn scikit-image opencv-python pandas scipy tqdm colorama tabulate einops einops-exts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 27.9 MB/s eta 0:00:00


# Tokens
*Authenticate with Hugging Face.*

In [4]:
import os, getpass

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token: ')

if not os.environ.get('WANDB_API_KEY'):
    key = getpass.getpass('W&B API key (leave empty for offline mode): ')
    if key:
        os.environ['WANDB_API_KEY'] = key
    else:
        os.environ['WANDB_MODE'] = 'offline'

W&B API key (leave empty for offline mode): ··········


# Drive
*Optionally mount Google Drive for persistence.*

In [5]:
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/wsi_anomaly_detection'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)

Mounted at /content/drive


# Dataset
*Fetch the Camelyon16-BMAD archive.*

In [6]:
os.makedirs('/content/gdrive_dataset', exist_ok=True)
!gdown --folder 'https://drive.google.com/drive/folders/1AC-wWZl_K18CWL2eIxUScoSOoxT4IBuw' -O /content/gdrive_dataset

Retrieving folder contents
Processing file 1xuapqiL1s18eMeKMd-Vi1Cqp2wp7Pghp Brain_AD.zip
Processing file 1SX24UkG3pYZfivy5kiBO7KzUPahGdAAC Chest-AD.zip
Processing file 1pXbfOgESrGhqZHaf2LGriH6TEG17uDDN Histopathology_AD.zip
Processing file 1sO4UaMvSKsbc8pgdnYCCnAO_iMSLUosh Liver_AD.zip
Processing file 1qsqVq5YYoKUFFkkUEbafD-BTYWKprfZY Retina_OCT2017_AD.zip
Processing file 1dPdONLpUcbIDoyDuHVI9t5b0HmpMM4Se Retina_RESC_AD.zip
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1xuapqiL1s18eMeKMd-Vi1Cqp2wp7Pghp
From (redirected): https://drive.google.com/uc?id=1xuapqiL1s18eMeKMd-Vi1Cqp2wp7Pghp&confirm=t&uuid=55910fc3-f7c5-4625-8fc0-317750f492bf
To: /content/gdrive_dataset/Brain_AD.zip
100% 401M/401M [00:10<00:00, 39.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1SX24UkG3pYZfivy5kiBO7KzUPahGdAAC
From (redirected): https://drive.google.com/uc?id=1SX24U

# Unzip
*Extract the dataset.*

In [7]:
import glob
zips = glob.glob('/content/**/Histopathology_AD.zip', recursive=True)

ZIP = zips[0]
EXTRACT = '/content/dataset'
os.makedirs(EXTRACT, exist_ok=True)
!unzip -q -o '{ZIP}' -d '{EXTRACT}'
DATA_ROOT = None
for root, dirs, files in os.walk(EXTRACT):
    if os.path.basename(root) == 'good' and os.path.basename(os.path.dirname(root)) == 'train':
        DATA_ROOT = os.path.dirname(os.path.dirname(root))
        break

In [12]:
print('DATA_ROOT =', DATA_ROOT)
print('train/good:', len(glob.glob(os.path.join(DATA_ROOT, 'train', 'good', '*.png'))),
      '| valid/good:', len(glob.glob(os.path.join(DATA_ROOT, 'valid', 'good', 'img', '*.png'))),
      '| valid/bad:', len(glob.glob(os.path.join(DATA_ROOT, 'valid', 'Ungood', 'img', '*.png'))),
      '| test/good:', len(glob.glob(os.path.join(DATA_ROOT, 'test', 'good', 'img', '*.png'))),
      '| test/bad:', len(glob.glob(os.path.join(DATA_ROOT, 'test', 'Ungood', 'img', '*.png'))))

DATA_ROOT = /content/dataset/camelyon16_256
train/good: 5088 | valid/good: 117 | valid/bad: 119 | test/good: 1003 | test/bad: 994


# CONCH Weights
*Download pretrained encoder weights.*

In [13]:
from huggingface_hub import login, hf_hub_download

login(token=os.environ['HF_TOKEN'])
os.environ['WEIGHTS_DIR'] = '/content/weights'
os.makedirs(os.environ['WEIGHTS_DIR'], exist_ok=True)
os.environ['CONCH_CKPT'] = hf_hub_download(repo_id='MahmoodLab/conch', filename='pytorch_model.bin', cache_dir=os.environ['WEIGHTS_DIR'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  802MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

# Config
*Generate the experiment configuration.*

In [14]:
import yaml
CONFIG = os.path.join(REPO, 'configs', 'conch', 'conch_bmad_config_ecr4ad.yaml')
SAVE_PATH = DRIVE_DIR if USE_DRIVE else '/content/results'
RUN_CONFIG = os.path.join(SAVE_PATH, 'run_config.yaml')  # archived next to the checkpoints

EPOCHS = 25
with open(CONFIG) as f:
    raw = yaml.safe_load(f)
raw['data_root']['value'] = DATA_ROOT
raw['save_path']['value'] = SAVE_PATH
raw['gpu_id']['value'] = '0'
n_train = len(glob.glob(os.path.join(DATA_ROOT, 'train', 'good', '*.png')))

batches = n_train // raw['batch_size']['value']

raw['total_iters']['value'] = EPOCHS * batches
raw['eval_period']['value'] = batches

os.makedirs(os.path.join(SAVE_PATH, 'weights'), exist_ok=True)
with open(RUN_CONFIG, 'w') as f:
    yaml.safe_dump(raw, f, sort_keys=False)

print('run config:', RUN_CONFIG)
print('train images:', n_train, '| batches/epoch:', batches)
print('total_iters:', raw['total_iters']['value'], '=', EPOCHS, 'epochs')

run config: /content/drive/MyDrive/wsi_anomaly_detection/run_config.yaml
train images: 5088 | batches/epoch: 159
total_iters: 3975 = 25 epochs


# Train
*Run training and evaluation.*

In [19]:
import os
os.chdir(REPO)
!python patch_train.py --config {RUN_CONFIG}

/content/wsi_anomaly_detection/dataloader/__init__.py:15: UserWarning: Whole-slide utilities (openslide/pyvips) are unavailable: No module named 'openslide'. Patch-level datasets (e.g. Camelyon16BMAD) will still work.
  warnings.warn(
/content/wsi_anomaly_detection/networks/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/content/wsi_anomaly_detection/networks/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/wsi_anomaly_detection/networks/dinov2/layers/attention.py:29: UserWarning: xFormers is disabled (Attention)
  warnings.warn("xFormers is disabled (Attention)")
/content/wsi_anomaly_detection/networks/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/content/wsi_anomaly_detection/networks/dinov2/layers/block.py:34: UserWarning: